In [1]:
import pandas as pd
import numpy as np

# pd.set_option('future.no_silent_downcasting', True):
# to prepare code for the future changes, opt-in with setting the future.no_silent_downcasting option
# ensure that Pandas code remains compatible with future versions, make code more robust and less likely to break when Pandas updates
pd.set_option('future.no_silent_downcasting', True)

csv_data = {'A': [1.0,5.0,10.0],
            'B': [2.0,6.0,11.0],
            'C': [3.0,'',12.0],
            'D': [4.0, 8.0, '']}

df = pd.DataFrame(csv_data)
df = df.replace('', np.nan)
df

,A,B,C,D
0,1.0,2.0,3.0,4.0
1,5.0,6.0,NaN,8.0
2,10.0,11.0,12.0,NaN


In [2]:
df.isnull().sum()

A    0
B    0
C    1
D    1
dtype: int64

In [3]:
df.values

array([[1.0, 2.0, 3.0, 4.0],
       [5.0, 6.0, nan, 8.0],
       [10.0, 11.0, 12.0, nan]], dtype=object)

In [4]:
df.dropna(axis=0)

,A,B,C,D
0,1.0,2.0,3.0,4.0


In [5]:
df

,A,B,C,D
0,1.0,2.0,3.0,4.0
1,5.0,6.0,NaN,8.0
2,10.0,11.0,12.0,NaN


In [6]:
df.dropna(axis=1)

,A,B
0,1.0,2.0
1,5.0,6.0
2,10.0,11.0


In [7]:
df.dropna(how='all')  

,A,B,C,D
0,1.0,2.0,3.0,4.0
1,5.0,6.0,NaN,8.0
2,10.0,11.0,12.0,NaN


In [8]:
# drop rows that have less than 3 real values 
df.dropna(thresh=4)

,A,B,C,D
0,1.0,2.0,3.0,4.0


In [9]:
# only drop rows where NaN appear in specific columns (here: 'C')
df.dropna(subset=['C'])

,A,B,C,D
0,1.0,2.0,3.0,4.0
2,10.0,11.0,12.0,NaN


In [10]:
# impute missing values via the column mean
import numpy as np
from sklearn.impute import SimpleImputer

imr = SimpleImputer(missing_values=np.nan, strategy='mean')
imr = imr.fit(df.values)  #using numpy array with .values
imputed_data = imr.transform(df.values)
imputed_data

array([[ 1. ,  2. ,  3. ,  4. ],
       [ 5. ,  6. ,  7.5,  8. ],
       [10. , 11. , 12. ,  6. ]])

In [11]:
df = pd.DataFrame([['green', 'M', 10.1, 'class1'],
                   ['red', 'L', 13.5, 'class2'],
                   ['blue', 'XL', 15.3, 'class1']])

df.columns = ['color', 'size', 'price', 'classlabel']
df

,color,size,price,classlabel
0,green,M,10.1,class1
1,red,L,13.5,class2
2,blue,XL,15.3,class1


In [12]:
size_mapping = {'XL': 3,
                'L': 2,
                'M': 1}

df['size'] = df['size'].map(size_mapping)
df

,color,size,price,classlabel
0,green,1,10.1,class1
1,red,2,13.5,class2
2,blue,3,15.3,class1


In [13]:
inv_size_mapping = {v: k for k, v in size_mapping.items()}
df['size'].map(inv_size_mapping)

0     M
1     L
2    XL
Name: size, dtype: object

In [14]:
class_mapping = {label: idx for idx, label in enumerate(np.unique(df['classlabel']))}
class_mapping

{'class1': 0, 'class2': 1}

In [15]:
df['classlabel'] = df['classlabel'].map(class_mapping)
df

,color,size,price,classlabel
0,green,1,10.1,0
1,red,2,13.5,1
2,blue,3,15.3,0


In [16]:
inv_class_mapping = {v: k for k, v in class_mapping.items()}
df['classlabel'] = df['classlabel'].map(inv_class_mapping)
df

,color,size,price,classlabel
0,green,1,10.1,class1
1,red,2,13.5,class2
2,blue,3,15.3,class1


In [17]:
from sklearn.preprocessing import LabelEncoder

# Label encoding with sklearn's LabelEncoder
class_le = LabelEncoder()
y = class_le.fit_transform(df['classlabel'].values)
y

array([0, 1, 0])

In [18]:
class_le.inverse_transform(y)

array(['class1', 'class2', 'class1'], dtype=object)

In [19]:
X = df[['color', 'size', 'price']].values

color_le = LabelEncoder()
X[:, 0] = color_le.fit_transform(X[:, 0])
X

array([[1, 1, 10.1],
       [2, 2, 13.5],
       [0, 3, 15.3]], dtype=object)

In [20]:
df = pd.DataFrame([['green', 'M', 10.1, 'class1'],
                   ['red', 'L', 13.5, 'class2'],
                   ['blue', 'XL', 15.3, 'class1']])

df.columns = ['color', 'size', 'price', 'classlabel']

X = df[['color', 'size', 'price']].values
X

array([['green', 'M', 10.1],
       ['red', 'L', 13.5],
       ['blue', 'XL', 15.3]], dtype=object)

In [21]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer

# Identify categorical columns
categorical_features_onehot = [0]  # Column 0 (color) gets one-hot encoding
categorical_features_ordinal = [1] # Column 1 (size) gets ordinal encoding
numerical_features = [2] # column 2 is numerical and will be passed through

# Create a ColumnTransformer
ct = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(sparse_output=False), categorical_features_onehot),
        ('ordinal', OrdinalEncoder(categories=[['M', 'L', 'XL']]), categorical_features_ordinal)
    ],
    remainder='passthrough'  # Pass through the remaining columns (numerical)
)

# Apply the transformation
X_transformed = ct.fit_transform(X)

# Print the transformed data
print(X_transformed)

[[0.0 1.0 0.0 0.0 10.1]
 [0.0 0.0 1.0 1.0 13.5]
 [1.0 0.0 0.0 2.0 15.3]]


In [22]:
df = pd.DataFrame([['green', 'M', 10.1, 'class1'],
                   ['red', 'L', 13.5, 'class2'],
                   ['blue', 'XL', 15.3, 'class1']])

df.columns = ['color', 'size', 'price', 'classlabel']

size_mapping = {'XL': 3,
                'L': 2,
                'M': 1}

df['size'] = df['size'].map(size_mapping)

df['color'] = df['color'].astype('category')

pd.get_dummies(df[['price', 'color', 'size']]).astype(int)

,price,size,color_blue,color_green,color_red
0,10,1,0,1,0
1,13,2,0,0,1
2,15,3,1,0,0


In [23]:
pd.get_dummies(df[['price', 'color', 'size']], drop_first=True).astype(int)

,price,size,color_green,color_red
0,10,1,1,0
1,13,2,0,1
2,15,3,0,0
